# QuEL-3 Deploy Check

This notebook validates QuEL-3 instrument deployment against your current quelware/clientware endpoint.

Choose `example_kind = "quel3"` for real deploy checks. You can also load the `quel1` example set to inspect configuration loading, but deploy/preflight cells will skip because instrument deployment is QuEL-3-specific.

It supports two connection modes:

- `use_standalone_client = False`: full quelware server mode (`ListUnits` is called)
- `use_standalone_client = True`: standalone worker/clientware mode (`ListUnits` is bypassed)


In [ ]:
from pathlib import Path

from qubex.backend.backend_controller import BACKEND_KIND_QUEL3
from qubex.backend.quel3.infra.quelware_imports import (
    import_module_with_workspace_fallback,
)
from qubex.core.async_bridge import DEFAULT_TIMEOUT_SECONDS, get_shared_async_bridge
from qubex.system.config_loader import ConfigLoader
from qubex.system.quel3 import Quel3ConfigurationManager

In [ ]:
# Example selection
example_kind = "quel3"
if example_kind not in {"quel1", "quel3"}:
    raise ValueError(f"Unsupported example_kind: {example_kind}")

candidate_root_dirs = [
    Path.cwd(),
    Path.cwd() / "docs/examples/system",
    Path.cwd().parent,
]
example_root = next(
    (
        path
        for path in candidate_root_dirs
        if (path / "quel1" / "config").is_dir()
        and (path / "quel3" / "config").is_dir()
    ),
    None,
)
if example_root is None:
    raise FileNotFoundError(
        "Could not find `docs/examples/system/{quel1,quel3}` from the current working directory."
    )

example_dir = example_root / example_kind
config_dir = example_dir / "config"
params_dir = example_dir / "params"

# Endpoint
server_host = "localhost"
server_port = 50051

# Set True when talking directly to a standalone worker/clientware endpoint.
use_standalone_client = True

# If None, derive unit labels from deploy requests.
standalone_unit_label: str | None = None

# Temporary workaround for current worker route-switch issue on non-readout ports.
# Use None to deploy all roles.
deploy_roles: set[str] | None = {"TRANSCEIVER"}

# ConfigLoader inputs
chip_id = "64Q"

# Target boxes. If empty, use all boxes in config.
box_ids: list[str] = []

print(f"example_root: {example_root.resolve()}")
print(f"example_kind: {example_kind}")
print(f"server: {server_host}:{server_port}")
print(f"use_standalone_client: {use_standalone_client}")
print(f"deploy_roles: {sorted(deploy_roles) if deploy_roles is not None else 'ALL'}")
print(f"chip_id: {chip_id}")
print(f"config_dir: {config_dir.resolve()}")
print(f"params_dir: {params_dir.resolve()}")


In [ ]:
loader = ConfigLoader(
    chip_id=chip_id,
    config_dir=config_dir,
    params_dir=params_dir,
    autoload=False,
)
loader.load()
experiment_system = loader.get_experiment_system()

configured_box_ids = [box.id for box in experiment_system.boxes]
if len(box_ids) == 0:
    box_ids = configured_box_ids

is_quel3_backend = loader.backend_kind == BACKEND_KIND_QUEL3

readout_targets_by_box: dict[str, list[str]] = {}
for label, target in sorted(experiment_system.gen_targets.items()):
    if target.type.value != "READ":
        continue
    readout_targets_by_box.setdefault(target.channel.port.box_id, []).append(label)

print("backend_kind:", loader.backend_kind)
print("boxes in system:", configured_box_ids)
print("boxes to deploy:", box_ids)
print("#gen_targets:", len(experiment_system.gen_targets))
print("deploy enabled:", is_quel3_backend)
print("readout targets by box:")
for box_id, labels in sorted(readout_targets_by_box.items()):
    print(f"- {box_id}: {tuple(labels)}")

if "T1" in readout_targets_by_box:
    t1_readout_targets = tuple(readout_targets_by_box["T1"])
    print("T1 readout target check:", t1_readout_targets == ("RQ00", "RQ01", "RQ02", "RQ03"))
else:
    t1_readout_targets = ()
    print("T1 not present in current example.")


In [ ]:
manager = Quel3ConfigurationManager(
    quelware_endpoint=server_host,
    quelware_port=server_port,
)

if is_quel3_backend:
    all_requests = manager._build_instrument_deploy_requests(  # noqa: SLF001
        experiment_system=experiment_system,
        box_ids=box_ids,
    )
    if deploy_roles is None:
        requests = all_requests
    else:
        requests = tuple(
            request for request in all_requests if request.role in deploy_roles
        )

    print("all deploy requests:", len(all_requests))
    print("filtered deploy requests:", len(requests))
    for request in requests:
        print(
            f"- port_id={request.port_id}, role={request.role}, "
            f"range=[{request.frequency_range_min_hz:.3e}, {request.frequency_range_max_hz:.3e}], "
            f"targets={request.target_labels}"
        )

    t1_requests = tuple(
        request for request in requests if request.port_id.startswith("quel3-02-a01:")
    )
    if t1_requests:
        print("T1 deploy request summary:")
        for request in t1_requests:
            print(
                f"- {request.port_id}: role={request.role}, targets={request.target_labels}"
            )
        t1_transceiver_targets = tuple(
            request.target_labels
            for request in t1_requests
            if request.role == "TRANSCEIVER"
        )
        print(
            "T1 transceiver target check:",
            t1_transceiver_targets == (("RQ00", "RQ01", "RQ02", "RQ03"),),
        )
else:
    requests = ()
    print("Loaded non-QuEL-3 example set. Deploy request preview is skipped.")


If you see `GRPCError(..., "Received :status = '400'", ...)` in full-server mode, the endpoint usually does not expose `SystemConfigurationService/ListUnits`.

In that case, switch `use_standalone_client = True` and re-run from the cells below.

If you see `GRPCError(..., "Received :status = '204'", ...)`, inspect the returned `grpc-status` / `grpc-message` headers. In a standalone setup this often indicates that the selected `unit_label` does not match a reachable upstream unit.

Current temporary workaround: `deploy_roles = {"TRANSCEIVER"}` deploys readout only and avoids the worker route-switch bug on non-readout ports. Set `deploy_roles = None` after that bug is fixed.

If `example_kind = "quel1"`, the cells below will skip because deploy is QuEL-3-specific.


In [ ]:
client_module = import_module_with_workspace_fallback("quelware_client.client")

create_quelware_client = client_module.create_quelware_client
create_standalone_client = client_module.create_standalone_client


def run_async(factory, *, timeout: float = DEFAULT_TIMEOUT_SECONDS):
    bridge = get_shared_async_bridge(key="quel3-deploy-check")
    return bridge.run(factory, timeout=timeout)


async def preflight_check():
    if not is_quel3_backend:
        print("Skipping preflight: loaded example is not QuEL-3.")
        return
    if len(requests) == 0:
        raise ValueError("No deploy requests were built.")

    if use_standalone_client:
        unit_label = standalone_unit_label or requests[0].port_id.split(":", 1)[0]
        async with create_standalone_client(
            server_host,
            server_port,
            unit_label=unit_label,
        ) as client:
            print("unit_labels:", client.list_unit_labels())
            resources = await client.list_resource_infos()
            print("resource_count:", len(resources))
            port_info = await client.get_port_info(requests[0].port_id)
            print("first_port:", port_info)
        return

    async with create_quelware_client(server_host, server_port) as client:
        print("unit_labels:", client.list_unit_labels())
        resources = await client.list_resource_infos()
        print("resource_count:", len(resources))
        port_info = await client.get_port_info(requests[0].port_id)
        print("first_port:", port_info)


run_async(preflight_check)


In [ ]:
instrument_module = import_module_with_workspace_fallback(
    "quelware_core.entities.instrument"
)

FixedTimelineProfile = instrument_module.FixedTimelineProfile
InstrumentDefinition = instrument_module.InstrumentDefinition
InstrumentMode = instrument_module.InstrumentMode
InstrumentRole = instrument_module.InstrumentRole


async def deploy_requests():
    if not is_quel3_backend:
        print("Skipping deploy: loaded example is not QuEL-3.")
        return {}

    deployed: dict[str, tuple[object, ...]] = {}

    if use_standalone_client:
        for request in requests:
            unit_label = standalone_unit_label or request.port_id.split(":", 1)[0]
            print(
                f"deploying alias={request.alias}, port_id={request.port_id}, targets={request.target_labels}"
            )
            async with create_standalone_client(
                server_host,
                server_port,
                unit_label=unit_label,
            ) as client:
                async with client.create_session([request.port_id]) as session:
                    profile = FixedTimelineProfile(
                        frequency_range_min=request.frequency_range_min_hz,
                        frequency_range_max=request.frequency_range_max_hz,
                    )
                    definition = InstrumentDefinition(
                        alias=request.alias,
                        mode=InstrumentMode.FIXED_TIMELINE,
                        role=getattr(InstrumentRole, request.role),
                        profile=profile,
                    )
                    inst_infos = await session.deploy_instruments(
                        request.port_id,
                        definitions=[definition],
                    )
                    deployed[request.alias] = tuple(inst_infos)
        return deployed

    async with create_quelware_client(server_host, server_port) as client:
        for request in requests:
            print(
                f"deploying alias={request.alias}, port_id={request.port_id}, targets={request.target_labels}"
            )
            async with client.create_session([request.port_id]) as session:
                profile = FixedTimelineProfile(
                    frequency_range_min=request.frequency_range_min_hz,
                    frequency_range_max=request.frequency_range_max_hz,
                )
                definition = InstrumentDefinition(
                    alias=request.alias,
                    mode=InstrumentMode.FIXED_TIMELINE,
                    role=getattr(InstrumentRole, request.role),
                    profile=profile,
                )
                inst_infos = await session.deploy_instruments(
                    request.port_id,
                    definitions=[definition],
                )
                deployed[request.alias] = tuple(inst_infos)
    return deployed


deployed = run_async(deploy_requests)
request_by_alias = {request.alias: request for request in requests}

print("deploy completed. aliases:", len(deployed))
for alias, infos in deployed.items():
    request = request_by_alias[alias]
    print(
        f"- {alias}: port_id={request.port_id}, targets={request.target_labels}, instrument_count={len(infos)}"
    )

for alias, request in request_by_alias.items():
    if request.port_id == "quel3-02-a01:trx_p00p01":
        print("T1 readout deploy confirmed for targets:", request.target_labels)
